# Unit Tests

This notebook is the regression checkpoint for the project.

- Run it after every code change.
- Add tests here whenever a new feature is implemented.
- Do not leave a feature without at least one test or smoke check.

In [ ]:
from pathlib import Path
import unittest
import socket
import tempfile
from unittest import mock

REPO_ROOT = Path.cwd()
print(f"Running notebook tests from: {REPO_ROOT}")

In [ ]:
from common.protocol import ProtocolError, decode_message, encode_message, make_message, receive_message, send_message
from common import message_types
from client.networking.client import ArenaClient
from client.state.controller import apply_server_message, return_to_lobby
from client.state.models import ClientAppState
from client.ui.pygame_client import challengeable_users, cycle_snake_color, default_login_form, invite_notice_color, is_reserved_movement_key, move_preview_snake, remaining_seconds, should_render_snake, snake_color_rgb, snake_head_color, game_over_reason_text, game_over_result_color, game_over_result_text, cheer_allowed, cheer_target_username, _handle_keydown, _handle_settings_click, LOBBY_LEFT_WIDTH, SETTINGS_FIELDS, SETTINGS_LEFT_X, SETTINGS_TOP_Y, WINDOW_WIDTH
from server.game import DOWN, LEFT, Match, RIGHT, UP
from server.lobby_manager import LobbyManager
from server.main import build_parser
from server.persistence import MatchHistoryStore
from server.server import PythonArenaServer
from server.session import UserSession
from server.user_registry import UserRegistry

class ProjectSmokeTests(unittest.TestCase):
    def test_agents_file_exists(self):
        self.assertTrue((REPO_ROOT / "AGENTS.md").exists())

    def test_project_pdf_exists(self):
        self.assertTrue((REPO_ROOT / "EECE350_project_Spring2025.pdf").exists())

    def test_plan_exists(self):
        self.assertTrue((REPO_ROOT / "plan.txt").exists())

    def test_notebook_runner_exists(self):
        self.assertTrue((REPO_ROOT / "tools" / "run_notebook.py").exists())

In [ ]:
class ProtocolTests(unittest.TestCase):
    def test_make_and_decode_round_trip(self):
        message = make_message(message_types.LOGIN, {"username": "Teo"})
        encoded = encode_message(message)
        body = encoded[4:]
        decoded = decode_message(body)
        self.assertEqual(decoded["type"], message_types.LOGIN)
        self.assertEqual(decoded["payload"]["username"], "Teo")

    def test_encode_requires_type(self):
        with self.assertRaises(ProtocolError):
            encode_message({"payload": {}})

    def test_receive_and_send_message_with_fake_socket(self):
        class FakeSocket:
            def __init__(self):
                self.buffer = bytearray()

            def sendall(self, data):
                self.buffer.extend(data)

            def recv(self, size):
                chunk = bytes(self.buffer[:size])
                del self.buffer[:size]
                return chunk

        fake_socket = FakeSocket()
        send_message(fake_socket, make_message(message_types.LOGIN_OK, {"username": "Rana"}))
        received = receive_message(fake_socket)
        self.assertEqual(received["type"], message_types.LOGIN_OK)
        self.assertEqual(received["payload"]["username"], "Rana")

In [ ]:
class ClientTests(unittest.TestCase):
    def test_login_includes_chat_port_when_provided(self):
        class FakeSocket:
            def __init__(self):
                self.buffer = bytearray()

            def sendall(self, data):
                self.buffer.extend(data)

            def recv(self, size):
                if not self.buffer:
                    response = encode_message(make_message(message_types.LOGIN_OK, {"username": "Teo"}))
                    self.buffer.extend(response)
                chunk = bytes(self.buffer[:size])
                del self.buffer[:size]
                return chunk

        client = ArenaClient()
        fake_socket = FakeSocket()
        client._socket = fake_socket
        recorded = {}

        def fake_send(sock, message):
            recorded.update(message)

        with mock.patch("client.networking.client.send_message", fake_send):
            with mock.patch("client.networking.client.receive_message", return_value={"type": message_types.LOGIN_OK, "payload": {}}):
                client.login("Teo", chat_port=7001)
        self.assertEqual(recorded["payload"]["chat_port"], 7001)

    def test_send_wraps_protocol_message(self):
        client = ArenaClient()
        client._socket = object()
        recorded = {}

        def fake_send(sock, message):
            recorded.update(message)

        with mock.patch("client.networking.client.send_message", fake_send):
            client.send(message_types.WAITING, {})
        self.assertEqual(recorded["type"], message_types.WAITING)

    def test_state_controller_applies_match_start(self):
        state = ClientAppState()
        apply_server_message(
            state,
            {
                "type": message_types.MATCH_START,
                "payload": {"state": {"players": ["Teo", "Joey"]}, "spectator": False},
            },
        )
        self.assertEqual(state.phase, "match")
        self.assertEqual(state.match_state["players"], ["Teo", "Joey"])

    def test_state_controller_stores_match_countdown(self):
        state = ClientAppState()
        apply_server_message(
            state,
            {
                "type": message_types.MATCH_START,
                "payload": {
                    "state": {"players": ["Teo", "Joey"]},
                    "spectator": False,
                    "countdown_seconds": 3,
                },
            },
        )
        self.assertEqual(state.countdown_seconds, 3)
        self.assertIsNone(state.countdown_end_ms)

    def test_state_controller_applies_game_over(self):
        state = ClientAppState(phase="match")
        apply_server_message(
            state,
            {
                "type": message_types.GAME_OVER,
                "payload": {"winner": "Teo", "state": {"reason": "timer_end"}},
            },
        )
        self.assertEqual(state.phase, "game_over")
        self.assertEqual(state.game_over["winner"], "Teo")

    def test_state_controller_keeps_login_phase_on_login_reject(self):
        state = ClientAppState(phase="login")
        apply_server_message(
            state,
            {
                "type": message_types.LOGIN_REJECT,
                "payload": {"message": "Username is invalid or already in use."},
            },
        )
        self.assertEqual(state.phase, "login")
        self.assertEqual(state.last_error, "Username is invalid or already in use.")

    def test_chat_request_messages_update_client_state(self):
        state = ClientAppState()
        apply_server_message(
            state,
            {
                "type": message_types.CHAT_REQUEST_SENT,
                "payload": {"target_username": "Joey", "expires_in_seconds": 60},
            },
        )
        self.assertEqual(state.outgoing_chat_request["target_username"], "Joey")
        apply_server_message(
            state,
            {
                "type": message_types.CHAT_REQUEST_RECEIVED,
                "payload": {"requester_username": "Rudy", "message": "Chat request from Rudy."},
            },
        )
        self.assertEqual(state.incoming_chat_request["requester_username"], "Rudy")

    def test_online_users_restores_pending_invite_state(self):
        state = ClientAppState(phase="game_over")
        apply_server_message(
            state,
            {
                "type": message_types.ONLINE_USERS,
                "payload": {
                    "users": ["Teo", "Joey"],
                    "waiting_players": [],
                    "pending_challenger": "Teo",
                    "outgoing_challenge_target": None,
                },
            },
        )
        self.assertEqual(state.challenger_username, "Teo")

    def test_challengeable_users_excludes_current_user(self):
        state = ClientAppState(username="Teo", online_users=["Teo", "Joey", "Rudy"])
        self.assertEqual(challengeable_users(state), ["Joey", "Rudy"])

    def test_default_login_form_uses_expected_defaults(self):
        form = default_login_form()
        self.assertEqual(form["host"], "127.0.0.1")
        self.assertEqual(form["port"], "5050")
        self.assertEqual(form["username"], "")
        self.assertEqual(form["chat_port"], "5051")

    def test_lobby_layout_leaves_right_third_empty(self):
        self.assertLessEqual(LOBBY_LEFT_WIDTH, int(WINDOW_WIDTH * 0.67))

    def test_remaining_seconds_converts_ticks_for_display(self):
        self.assertEqual(remaining_seconds({"remaining_ticks": 4}), 1)
        self.assertEqual(remaining_seconds({"remaining_ticks": 11}), 2)
        self.assertEqual(remaining_seconds({"remaining_ticks": 420}), 60)

    def test_should_render_snake_blinks_on_stun_ticks(self):
        self.assertFalse(should_render_snake({"stun_ticks_remaining": 5}))
        self.assertFalse(should_render_snake({"stun_ticks_remaining": 3}))
        self.assertTrue(should_render_snake({"stun_ticks_remaining": 4}))
        self.assertTrue(should_render_snake({}))

    def test_invite_notice_color_alternates_between_neon_colors(self):
        self.assertEqual(invite_notice_color(0), (255, 60, 190))
        self.assertEqual(invite_notice_color(250), (0, 214, 255))
        self.assertEqual(invite_notice_color(500), (255, 60, 190))

    def test_snake_color_rgb_uses_named_presets(self):
        self.assertEqual(snake_color_rgb("pink"), (255, 60, 190))
        self.assertEqual(snake_color_rgb("blue"), (0, 214, 255))
        self.assertEqual(snake_color_rgb("purple"), (170, 0, 255))

    def test_snake_head_color_has_extra_contrast_for_green_and_yellow(self):
        self.assertNotEqual(snake_head_color((57, 255, 20)), (57, 255, 20))
        self.assertNotEqual(snake_head_color((255, 240, 0)), (255, 240, 0))
        self.assertGreater(sum(snake_head_color((57, 255, 20))), sum((57, 255, 20)))

    def test_snake_color_rgb_matches_user_label_color_source(self):
        self.assertEqual(snake_color_rgb("blue"), (0, 214, 255))
        self.assertEqual(snake_color_rgb("pink"), (255, 60, 190))

    def test_cycle_snake_color_wraps_around_presets(self):
        self.assertEqual(cycle_snake_color("pink", 1), "blue")
        self.assertEqual(cycle_snake_color("pink", -1), "red")

    def test_move_preview_snake_updates_preview_body(self):
        state = ClientAppState()
        original = list(state.preview_body)
        move_preview_snake(state, "RIGHT")
        self.assertNotEqual(state.preview_body, original)
        self.assertEqual(state.preview_direction, "RIGHT")

    def test_settings_back_click_returns_to_lobby(self):
        class FakeClient:
            def send(self, *_args, **_kwargs):
                return None

        class FakeRect:
            def __init__(self, x, y, w, h):
                self.x = x
                self.y = y
                self.w = w
                self.h = h

            def collidepoint(self, x, y):
                return self.x <= x <= self.x + self.w and self.y <= y <= self.y + self.h

        class FakePygame:
            Rect = FakeRect

        state = ClientAppState(phase="settings")
        state.settings_field_index = len(SETTINGS_FIELDS) - 1
        row_y = SETTINGS_TOP_Y + 120 + (4 * 70)
        _handle_settings_click(FakeClient(), state, FakePygame(), (SETTINGS_LEFT_X + 10, row_y + 20))
        self.assertEqual(state.phase, "lobby")

    def test_reserved_movement_keys_are_rejected(self):
        self.assertTrue(is_reserved_movement_key(13))
        self.assertTrue(is_reserved_movement_key(27))
        self.assertTrue(is_reserved_movement_key(49))
        self.assertTrue(is_reserved_movement_key(50))
        self.assertFalse(is_reserved_movement_key(97))
        self.assertFalse(is_reserved_movement_key(1073741906))

    def test_settings_rebind_rejects_reserved_command_keys(self):
        class FakeClient:
            def send(self, *_args, **_kwargs):
                return None

        class FakePygame:
            K_ESCAPE = 27
            K_RETURN = 13
            K_KP_ENTER = 1073741912
            K_BACKSPACE = 8
            K_UP = 1073741906
            K_DOWN = 1073741905
            K_LEFT = 1073741904
            K_RIGHT = 1073741903

        class FakeEvent:
            def __init__(self, key, unicode=""):
                self.key = key
                self.unicode = unicode

        state = ClientAppState(phase="settings")
        state.rebinding_direction = "UP"
        original = state.movement_keys["UP"]
        result = _handle_keydown(
            FakeClient(),
            state,
            FakePygame(),
            FakeEvent(13),
            {},
            "connect",
            0,
            lambda: None,
            None,
        )
        self.assertEqual(result, 0)
        self.assertEqual(state.movement_keys["UP"], original)
        self.assertEqual(state.rebinding_direction, "UP")
        self.assertEqual(state.last_error, "ENTER is reserved for commands.")

    def test_cheer_allowed_enforces_cooldown(self):
        self.assertTrue(cheer_allowed(None, 1000))
        self.assertFalse(cheer_allowed(1000, 1500))
        self.assertTrue(cheer_allowed(1000, 2100))

    def test_cheer_target_username_uses_selected_player_slot(self):
        match_state = {
            "snakes": {
                "Left": {"body": [(2, 3), (1, 3), (0, 3)], "color": "blue"},
                "Right": {"body": [(8, 9), (8, 8), (8, 7)], "color": "pink"},
            }
        }
        self.assertEqual(cheer_target_username(match_state, 0), "Left")
        self.assertEqual(cheer_target_username(match_state, 1), "Right")
        self.assertIsNone(cheer_target_username(match_state, 2))

    def test_game_over_result_text_is_personalized(self):
        winning = ClientAppState(username="Teo", phase="game_over", game_over={"winner": "Teo"})
        losing = ClientAppState(username="Teo", phase="game_over", game_over={"winner": "Joey"})
        spectator = ClientAppState(username="Fan", phase="game_over", spectator=True, game_over={"winner": "Joey"})
        tied = ClientAppState(username="Teo", phase="game_over", game_over={"winner": None})
        self.assertEqual(game_over_result_text(winning), "You won!")
        self.assertEqual(game_over_result_text(losing), "You lost.")
        self.assertEqual(game_over_result_text(spectator), "Joey won!")
        self.assertEqual(game_over_result_text(tied), "You tied.")

    def test_game_over_result_color_matches_outcome(self):
        winning = ClientAppState(username="Teo", phase="game_over", game_over={"winner": "Teo"})
        losing = ClientAppState(username="Teo", phase="game_over", game_over={"winner": "Joey"})
        spectator = ClientAppState(
            username="Fan",
            phase="game_over",
            spectator=True,
            game_over={
                "winner": "Joey",
                "state": {"snakes": {"Joey": {"color": "blue"}}},
            },
        )
        self.assertEqual(game_over_result_color(winning), (57, 255, 20))
        self.assertEqual(game_over_result_color(losing), (255, 40, 40))
        self.assertEqual(game_over_result_color(spectator), (0, 214, 255))

    def test_game_over_reason_text_is_human_friendly(self):
        self.assertEqual(game_over_reason_text("health_zero"), "Somebody ran out of snake juice.")
        self.assertEqual(game_over_reason_text("timer_end"), "Time called. The healthier serpent survived.")
        self.assertEqual(game_over_reason_text("player_disconnected"), "One snake vanished into the void.")

    def test_return_to_lobby_clears_match_ui_state(self):
        state = ClientAppState(
            phase="game_over",
            match_state={"players": ["Teo", "Joey"]},
            game_over={"winner": "Teo"},
            spectator=True,
            disconnected_player="Joey",
            challenger_username="Rudy",
            outgoing_challenge_target="Joey",
            last_error="oops",
        )
        return_to_lobby(state)
        self.assertEqual(state.phase, "lobby")
        self.assertIsNone(state.match_state)
        self.assertIsNone(state.game_over)
        self.assertFalse(state.spectator)
        self.assertIsNone(state.disconnected_player)
        self.assertEqual(state.challenger_username, "Rudy")
        self.assertEqual(state.outgoing_challenge_target, "Joey")
        self.assertIsNone(state.last_error)

In [ ]:
class UserRegistryTests(unittest.TestCase):
    def test_register_rejects_duplicate_usernames(self):
        registry = UserRegistry()
        self.assertTrue(registry.register("Teo", object()))
        self.assertFalse(registry.register("Teo", object()))

    def test_register_rejects_blank_usernames(self):
        registry = UserRegistry()
        self.assertFalse(registry.register("   ", object()))

    def test_unregister_removes_username(self):
        registry = UserRegistry()
        registry.register("Joey", object())
        registry.unregister("Joey")
        self.assertFalse(registry.is_taken("Joey"))

    def test_list_usernames_returns_sorted_names(self):
        registry = UserRegistry()
        registry.register("Rudy", object())
        registry.register("Teo", object())
        self.assertEqual(registry.list_usernames(), ["Rudy", "Teo"])

In [ ]:
class LobbyManagerTests(unittest.TestCase):
    def test_chat_request_replaces_previous_outgoing_request(self):
        lobby = LobbyManager()
        success, _message, canceled = lobby.issue_chat_request(
            "Teo", "Joey", "127.0.0.1", 7001, {"Teo", "Joey", "Rudy"}, ttl_seconds=60
        )
        self.assertTrue(success)
        self.assertIsNone(canceled)
        success, _message, canceled = lobby.issue_chat_request(
            "Teo", "Rudy", "127.0.0.1", 7001, {"Teo", "Joey", "Rudy"}, ttl_seconds=60
        )
        self.assertTrue(success)
        self.assertEqual(canceled["target_username"], "Joey")
        self.assertEqual(lobby.pending_chat_request_for("Rudy")["requester_username"], "Teo")

    def test_chat_request_accept_returns_peer_connection_info(self):
        lobby = LobbyManager()
        lobby.issue_chat_request("Teo", "Joey", "127.0.0.1", 7001, {"Teo", "Joey"}, ttl_seconds=60)
        success, _message, request = lobby.accept_chat_request("Joey", "Teo")
        self.assertTrue(success)
        self.assertEqual(request["requester_host"], "127.0.0.1")
        self.assertEqual(request["requester_port"], 7001)

    def test_waiting_players_are_sorted(self):
        lobby = LobbyManager()
        lobby.set_waiting("Rudy")
        lobby.set_waiting("Teo")
        self.assertEqual(lobby.waiting_players(), ["Rudy", "Teo"])

    def test_issue_challenge_rejects_self_target(self):
        lobby = LobbyManager()
        success, message = lobby.issue_challenge("Teo", "Teo", {"Teo"})
        self.assertFalse(success)
        self.assertIn("cannot invite yourself", message)

    def test_issue_challenge_tracks_pending_request(self):
        lobby = LobbyManager()
        success, _ = lobby.issue_challenge("Teo", "Joey", {"Teo", "Joey"})
        self.assertTrue(success)
        accepted, _ = lobby.accept_challenge("Joey", "Teo")
        self.assertTrue(accepted)

    def test_issue_challenge_rejects_second_pending_target(self):
        lobby = LobbyManager()
        success, _ = lobby.issue_challenge("Teo", "Joey", {"Teo", "Joey", "Rudy"})
        self.assertTrue(success)
        success, message = lobby.issue_challenge("Rudy", "Joey", {"Teo", "Joey", "Rudy"})
        self.assertFalse(success)
        self.assertIn("already has a pending invite", message)

    def test_accept_rejects_missing_request(self):
        lobby = LobbyManager()
        success, message = lobby.accept_challenge("Joey", "Teo")
        self.assertFalse(success)
        self.assertIn("No matching invite", message)

    def test_clear_player_removes_waiting_and_pending_state(self):
        lobby = LobbyManager()
        lobby.set_waiting("Teo")
        lobby.issue_challenge("Joey", "Teo", {"Teo", "Joey"})
        lobby.clear_player("Teo")
        success, _ = lobby.accept_challenge("Teo", "Joey")
        self.assertFalse(success)
        self.assertEqual(lobby.waiting_players(), [])

    def test_restore_challenge_recreates_pending_request(self):
        lobby = LobbyManager()
        lobby.issue_challenge("Joey", "Teo", {"Teo", "Joey"})
        accepted, _ = lobby.accept_challenge("Teo", "Joey")
        self.assertTrue(accepted)
        lobby.restore_challenge("Teo", "Joey")
        self.assertEqual(lobby.pending_challenger_for("Teo"), "Joey")

    def test_second_outgoing_invite_replaces_previous_target(self):
        lobby = LobbyManager()
        success, _ = lobby.issue_challenge("Teo", "Joey", {"Teo", "Joey", "Rudy"})
        self.assertTrue(success)
        success, message = lobby.issue_challenge("Teo", "Rudy", {"Teo", "Joey", "Rudy"})
        self.assertTrue(success)
        self.assertIn("invited Rudy", message)
        self.assertIsNone(lobby.pending_challenger_for("Joey"))
        self.assertEqual(lobby.pending_challenger_for("Rudy"), "Teo")

    def test_reciprocal_invite_is_rejected_when_invite_is_already_pending(self):
        lobby = LobbyManager()
        success, _ = lobby.issue_challenge("Teo", "Joey", {"Teo", "Joey"})
        self.assertTrue(success)
        success, message = lobby.issue_challenge("Joey", "Teo", {"Teo", "Joey"})
        self.assertFalse(success)
        self.assertIn("pending invite from this player", message)
        self.assertEqual(lobby.pending_challenger_for("Joey"), "Teo")

    def test_pending_target_for_returns_current_outgoing_target(self):
        lobby = LobbyManager()
        lobby.issue_challenge("Teo", "Joey", {"Teo", "Joey"})
        self.assertEqual(lobby.pending_target_for("Teo"), "Joey")
        self.assertIsNone(lobby.pending_target_for("Joey"))

    def test_clear_all_invites_removes_pending_invites_only(self):
        lobby = LobbyManager()
        lobby.set_waiting("Rudy")
        lobby.issue_challenge("Teo", "Joey", {"Teo", "Joey", "Rudy"})
        lobby.clear_all_invites()
        self.assertIsNone(lobby.pending_challenger_for("Joey"))
        self.assertEqual(lobby.waiting_players(), ["Rudy"])

In [ ]:
class CliTests(unittest.TestCase):
    def test_server_parser_accepts_positional_port(self):
        parser = build_parser()
        args = parser.parse_args(["5051"])
        self.assertEqual(args.port, 5051)

    def test_server_parser_accepts_flag_port(self):
        parser = build_parser()
        args = parser.parse_args(["--port", "5052"])
        self.assertEqual(args.port_flag, 5052)

    def test_client_parser_accepts_pygame_mode(self):
        from client.main import build_parser as build_client_parser
        parser = build_client_parser()
        args = parser.parse_args(["--mode", "pygame"])
        self.assertEqual(args.mode, "pygame")

    def test_client_parser_defaults_to_pygame(self):
        from client.main import build_parser as build_client_parser
        parser = build_client_parser()
        args = parser.parse_args([])
        self.assertEqual(args.mode, "pygame")

In [ ]:
class MatchTests(unittest.TestCase):
    def test_match_initializes_two_snakes(self):
        match = Match(players=["Teo", "Joey"])
        self.assertEqual(sorted(match.snakes.keys()), ["Joey", "Teo"])
        self.assertEqual(match.snakes["Teo"].direction, RIGHT)
        self.assertEqual(match.snakes["Joey"].direction, LEFT)

    def test_tick_moves_snakes_forward(self):
        match = Match(players=["Teo", "Joey"])
        state = match.tick()
        self.assertEqual(state["snakes"]["Teo"]["body"][0], [5, 10])
        self.assertEqual(state["snakes"]["Joey"]["body"][0], [24, 10])

    def test_queue_input_changes_direction_on_tick(self):
        match = Match(players=["Teo", "Joey"])
        self.assertTrue(match.queue_input("Teo", UP))
        match.tick()
        self.assertEqual(match.snakes["Teo"].direction, UP)
        self.assertEqual(match.snakes["Teo"].body[0], (4, 9))

    def test_reverse_direction_is_ignored(self):
        match = Match(players=["Teo", "Joey"])
        self.assertTrue(match.queue_input("Teo", LEFT))
        match.tick()
        self.assertEqual(match.snakes["Teo"].direction, RIGHT)
        self.assertEqual(match.snakes["Teo"].body[0], (5, 10))

    def test_match_becomes_game_over_when_timer_ends(self):
        match = Match(players=["Teo", "Joey"], remaining_ticks=1)
        state = match.tick()
        self.assertTrue(state["game_over"])
        self.assertIsNone(state["winner"])

    def test_higher_health_wins_when_timer_ends(self):
        match = Match(players=["Teo", "Joey"], remaining_ticks=1)
        match.snakes["Teo"].health = 120
        match.snakes["Joey"].health = 80
        state = match.tick()
        self.assertEqual(state["winner"], "Teo")

    def test_wall_collision_reduces_health_and_keeps_body_in_bounds(self):
        match = Match(players=["Teo", "Joey"], board_width=6, board_height=6, obstacles=[])
        match.snakes["Teo"].body = [(5, 2), (4, 2), (3, 2)]
        match.snakes["Teo"].direction = RIGHT
        state = match.tick()
        self.assertEqual(state["snakes"]["Teo"]["health"], 85)
        self.assertEqual(state["snakes"]["Teo"]["body"][0], [5, 2])

    def test_obstacle_collision_reduces_health(self):
        match = Match(players=["Teo", "Joey"], obstacles=[(5, 10)])
        state = match.tick()
        self.assertEqual(state["snakes"]["Teo"]["health"], 90)
        self.assertEqual(state["snakes"]["Teo"]["body"][0], [4, 10])

    def test_body_collision_can_end_match(self):
        match = Match(players=["Teo", "Joey"], obstacles=[])
        match.snakes["Teo"].health = 20
        match.snakes["Teo"].body = [(4, 4), (4, 5), (5, 5), (5, 4)]
        match.snakes["Teo"].direction = DOWN
        state = match.tick()
        self.assertTrue(state["game_over"])
        self.assertEqual(state["winner"], "Joey")

    def test_head_on_collision_only_applies_head_on_damage(self):
        match = Match(players=["Teo", "Joey"], obstacles=[])
        match.snakes["Teo"].body = [(4, 4), (3, 4), (2, 4)]
        match.snakes["Teo"].direction = RIGHT
        match.snakes["Joey"].body = [(6, 4), (7, 4), (8, 4)]
        match.snakes["Joey"].direction = LEFT
        state = match.tick()
        self.assertEqual(state["snakes"]["Teo"]["health"], 85)
        self.assertEqual(state["snakes"]["Joey"]["health"], 85)

    def test_collecting_green_pie_increases_health(self):
        match = Match(players=["Teo", "Joey"], obstacles=[], pies=[{"x": 5, "y": 10, "kind": "green"}])
        match.snakes["Teo"].health = 95
        state = match.tick()
        self.assertEqual(state["snakes"]["Teo"]["health"], 100)

    def test_collecting_gold_pie_still_increases_health(self):
        match = Match(players=["Teo", "Joey"], obstacles=[], pies=[{"x": 5, "y": 10, "kind": "gold"}])
        match.snakes["Teo"].health = 95
        state = match.tick()
        self.assertEqual(state["snakes"]["Teo"]["health"], 100)

    def test_collecting_pie_does_not_change_snake_length(self):
        match = Match(players=["Teo", "Joey"], obstacles=[], pies=[{"x": 5, "y": 10, "kind": "green"}])
        state = match.tick()
        self.assertEqual(len(match.snakes["Teo"].body), 3)
        self.assertEqual(len(state["snakes"]["Teo"]["body"]), 3)

    def test_match_restocks_one_random_green_pie(self):
        match = Match(players=["Teo", "Joey"], obstacles=[], pies=[])
        self.assertEqual(len(match.pies), 1)
        pie = match.pies[0]
        self.assertEqual(pie["kind"], "green")
        occupied = set(match.obstacles)
        for snake in match.snakes.values():
            occupied.update(snake.body)
        self.assertNotIn((pie["x"], pie["y"]), occupied)

    def test_default_obstacles_avoid_initial_spawn_zones(self):
        match = Match(players=["Teo", "Joey"], pies=[])
        safe_cells = match._spawn_safe_cells()
        self.assertGreater(len(match.obstacles), 0)
        for obstacle in match.obstacles:
            self.assertNotIn(obstacle, safe_cells)

    def test_match_restocks_minimum_pies(self):
        match = Match(players=["Teo", "Joey"], obstacles=[], pies=[])
        self.assertGreaterEqual(len(match.pies), 1)
        match.pies.clear()
        match.tick()
        self.assertGreaterEqual(len(match.pies), 1)

    def test_collision_freezes_then_turns_snake_away(self):
        match = Match(players=["Teo", "Joey"], board_width=6, board_height=6, obstacles=[])
        match.snakes["Teo"].body = [(5, 3), (4, 3), (3, 3)]
        match.snakes["Teo"].direction = RIGHT
        first = match.tick()
        self.assertEqual(first["snakes"]["Teo"]["health"], 85)
        self.assertEqual(first["snakes"]["Teo"]["body"][0], [5, 3])
        for _ in range(4):
            frozen = match.tick()
            self.assertEqual(frozen["snakes"]["Teo"]["body"][0], [5, 3])
            self.assertEqual(frozen["snakes"]["Teo"]["body"], [[5, 3], [4, 3], [3, 3]])
        recovered = match.tick()
        self.assertEqual(recovered["snakes"]["Teo"]["body"][0], [5, 2])

    def test_add_cheer_keeps_recent_messages_only(self):
        match = Match(players=["Teo", "Joey"])
        for idx in range(35):
            match.add_cheer("Teo", f"msg-{idx}")
        self.assertEqual(len(match.cheers), 30)
        self.assertEqual(match.cheers[0]["text"], "msg-5")

In [ ]:
class PersistenceTests(unittest.TestCase):
    def test_match_history_store_saves_match_summary(self):
        store = MatchHistoryStore(":memory:")
        match = Match(players=["Teo", "Joey"], remaining_ticks=1)
        match.add_cheer("Teo", "Go")
        state = match.tick()
        store.save_match(state)
        rows = store.list_recent_matches()
        self.assertEqual(len(rows), 1)
        self.assertEqual(rows[0]["player_one"], "Teo")
        self.assertEqual(rows[0]["player_two"], "Joey")

In [ ]:
class ServerFlowTests(unittest.TestCase):
    def test_start_match_rejects_replacing_active_match(self):
        server = PythonArenaServer("127.0.0.1", 5050, db_path=":memory:")
        server._running.set()
        server.active_match = Match(players=["Teo", "Joey"])
        started, reason = server.start_match(["Rudy", "Rana"])
        self.assertFalse(started)
        self.assertEqual(reason, "busy")
        self.assertEqual(server.active_match.players, ["Teo", "Joey"])
        server._running.clear()

    def test_start_match_rejects_missing_player_session(self):
        class FakeSocket:
            def sendall(self, data):
                return None

            def close(self):
                return None

        server = PythonArenaServer("127.0.0.1", 5050, db_path=":memory:")
        server.user_registry.register("Teo", UserSession(address=("127.0.0.1", 1), socket=FakeSocket(), username="Teo"))
        started, reason = server.start_match(["Teo", "Joey"])
        self.assertFalse(started)
        self.assertEqual(reason, "offline")
        self.assertIsNone(server.active_match)

    def test_start_match_creates_fresh_thread(self):
        class FakeSocket:
            def sendall(self, data):
                return None

            def close(self):
                return None

        class FakeThread:
            instances = []

            def __init__(self, target=None, args=(), daemon=None):
                self.target = target
                self.args = args
                self.daemon = daemon
                self.started = False
                FakeThread.instances.append(self)

            def start(self):
                self.started = True

        server = PythonArenaServer("127.0.0.1", 5050, db_path=":memory:")
        for idx, username in enumerate(["Teo", "Joey"], start=1):
            server.user_registry.register(username, UserSession(address=("127.0.0.1", idx), socket=FakeSocket(), username=username))
        previous_thread = FakeThread()
        server._match_thread = previous_thread
        with mock.patch("server.server.threading.Thread", FakeThread):
            started, reason = server.start_match(["Teo", "Joey"])
        self.assertTrue(started)
        self.assertIsNone(reason)
        self.assertIsNot(server._match_thread, previous_thread)
        self.assertTrue(server._match_thread.started)

    def test_start_match_clears_all_pending_invites(self):
        class FakeSocket:
            def sendall(self, data):
                return None

            def close(self):
                return None

        server = PythonArenaServer("127.0.0.1", 5050, db_path=":memory:")
        for idx, username in enumerate(["Teo", "Joey", "Rudy"], start=1):
            server.user_registry.register(username, UserSession(address=("127.0.0.1", idx), socket=FakeSocket(), username=username))
        server.lobby_manager.issue_challenge("Rudy", "Teo", {"Teo", "Joey", "Rudy"})
        server.lobby_manager.issue_challenge("Joey", "Rudy", {"Teo", "Joey", "Rudy"})
        started, reason = server.start_match(["Teo", "Joey"])
        self.assertTrue(started)
        self.assertIsNone(reason)
        self.assertIsNone(server.lobby_manager.pending_challenger_for("Teo"))
        self.assertIsNone(server.lobby_manager.pending_challenger_for("Rudy"))

    def test_run_match_loop_does_not_clear_newer_match(self):
        server = PythonArenaServer("127.0.0.1", 5050, db_path=":memory:")
        server._running.set()
        old_match = Match(players=["Teo", "Joey"], remaining_ticks=1)
        new_match = Match(players=["Rudy", "Rana"])
        server.active_match = old_match
        old_match.tick()
        server.active_match = new_match
        server.run_match_loop(old_match)
        self.assertIs(server.active_match, new_match)
        server._running.clear()

    def test_start_match_does_not_auto_send_match_start_to_nonwatchers(self):
        class FakeSocket:
            def __init__(self):
                self.buffer = []

            def sendall(self, data):
                self.buffer.append(data)

            def close(self):
                return None

        server = PythonArenaServer("127.0.0.1", 5050, db_path=":memory:")
        teo_session = UserSession(address=("127.0.0.1", 1), socket=FakeSocket(), username="Teo")
        joey_session = UserSession(address=("127.0.0.1", 2), socket=FakeSocket(), username="Joey")
        rudy_session = UserSession(address=("127.0.0.1", 3), socket=FakeSocket(), username="Rudy")
        server.user_registry.register("Teo", teo_session)
        server.user_registry.register("Joey", joey_session)
        server.user_registry.register("Rudy", rudy_session)
        server.add_spectator("Rudy")
        started, reason = server.start_match(["Teo", "Joey"])
        self.assertTrue(started)
        self.assertIsNone(reason)
        self.assertGreaterEqual(len(teo_session.socket.buffer), 1)
        self.assertGreaterEqual(len(joey_session.socket.buffer), 1)
        self.assertEqual(len(rudy_session.socket.buffer), 0)

    def test_disconnect_ends_active_match_and_selects_remaining_player(self):
        class FakeSocket:
            def __init__(self):
                self.buffer = []

            def sendall(self, data):
                self.buffer.append(data)

            def close(self):
                return None

        server = PythonArenaServer("127.0.0.1", 5050, db_path=":memory:")
        teo_session = UserSession(address=("127.0.0.1", 1), socket=FakeSocket(), username="Teo")
        joey_session = UserSession(address=("127.0.0.1", 2), socket=FakeSocket(), username="Joey")
        server.user_registry.register("Teo", teo_session)
        server.user_registry.register("Joey", joey_session)
        server.active_match = Match(players=["Teo", "Joey"])
        server.handle_player_disconnect("Teo")
        self.assertIsNone(server.active_match)
        self.assertGreaterEqual(len(joey_session.socket.buffer), 2)
        player_disconnected = decode_message(joey_session.socket.buffer[0][4:])
        game_over = decode_message(joey_session.socket.buffer[1][4:])
        self.assertEqual(player_disconnected["type"], message_types.PLAYER_DISCONNECTED)
        self.assertEqual(game_over["type"], message_types.GAME_OVER)
        self.assertEqual(game_over["payload"]["winner"], "Joey")

    def test_disconnect_notifies_spectators_too(self):
        class FakeSocket:
            def __init__(self):
                self.buffer = []

            def sendall(self, data):
                self.buffer.append(data)

            def close(self):
                return None

        server = PythonArenaServer("127.0.0.1", 5050, db_path=":memory:")
        teo_session = UserSession(address=("127.0.0.1", 1), socket=FakeSocket(), username="Teo")
        joey_session = UserSession(address=("127.0.0.1", 2), socket=FakeSocket(), username="Joey")
        sara_session = UserSession(address=("127.0.0.1", 3), socket=FakeSocket(), username="Sara")
        server.user_registry.register("Teo", teo_session)
        server.user_registry.register("Joey", joey_session)
        server.user_registry.register("Sara", sara_session)
        server.add_spectator("Sara")
        server.active_match = Match(players=["Teo", "Joey"])
        server.handle_player_disconnect("Teo")
        self.assertGreaterEqual(len(sara_session.socket.buffer), 2)
        spectator_disconnect = decode_message(sara_session.socket.buffer[0][4:])
        spectator_game_over = decode_message(sara_session.socket.buffer[1][4:])
        self.assertEqual(spectator_disconnect["type"], message_types.PLAYER_DISCONNECTED)
        self.assertEqual(spectator_game_over["type"], message_types.GAME_OVER)
        self.assertEqual(spectator_game_over["payload"]["winner"], "Joey")

    def test_handle_client_allows_login_retry_without_reconnect(self):
        class FakeSocket:
            def close(self):
                return None

        server = PythonArenaServer("127.0.0.1", 5050, db_path=":memory:")
        server._running.set()
        outbound = []

        messages = iter([
            make_message(message_types.LOGIN, {"username": "   "}),
            make_message(message_types.LOGIN, {"username": "Teo"}),
        ])

        def fake_receive_message(sock):
            try:
                return next(messages)
            except StopIteration:
                raise ConnectionError()

        def fake_send_message(sock, message):
            outbound.append(message)

        with mock.patch("server.server.receive_message", side_effect=fake_receive_message):
            with mock.patch("server.server.send_message", side_effect=fake_send_message):
                with mock.patch.object(server, "broadcast_online_users"):
                    server.handle_client(FakeSocket(), ("127.0.0.1", 12345))

        message_types_sent = [message["type"] for message in outbound]
        self.assertIn(message_types.LOGIN_REJECT, message_types_sent)
        self.assertIn(message_types.LOGIN_OK, message_types_sent)
        self.assertLess(message_types_sent.index(message_types.LOGIN_REJECT), message_types_sent.index(message_types.LOGIN_OK))

    def test_handle_settings_update_stores_session_color(self):
        class FakeSocket:
            def __init__(self):
                self.buffer = []

            def sendall(self, data):
                self.buffer.append(data)

            def close(self):
                return None

        server = PythonArenaServer("127.0.0.1", 5050, db_path=":memory:")
        session = UserSession(address=("127.0.0.1", 1), socket=FakeSocket(), username="Teo")
        server.handle_settings_update(session, {"snake_color": "purple"})
        self.assertEqual(session.snake_color, "purple")
        response = decode_message(session.socket.buffer[0][4:])
        self.assertEqual(response["type"], message_types.SETTINGS_UPDATE)
        self.assertEqual(response["payload"]["snake_color"], "purple")

    def test_busy_server_preserves_pending_challenge(self):
        class FakeSocket:
            def __init__(self):
                self.buffer = []

            def sendall(self, data):
                self.buffer.append(data)

            def close(self):
                return None

        server = PythonArenaServer("127.0.0.1", 5050, db_path=":memory:")
        server.active_match = Match(players=["HostA", "HostB"])
        joey_session = UserSession(address=("127.0.0.1", 2), socket=FakeSocket(), username="Joey")
        teo_session = UserSession(address=("127.0.0.1", 1), socket=FakeSocket(), username="Teo")
        server.user_registry.register("Teo", teo_session)
        server.user_registry.register("Joey", joey_session)
        server.lobby_manager.issue_challenge("Joey", "Teo", {"Teo", "Joey"})
        server.handle_challenge_accept(teo_session, {"challenger_username": "Joey"})
        self.assertEqual(server.lobby_manager.pending_challenger_for("Teo"), "Joey")

    def test_target_notification_failure_does_not_disconnect_challenger(self):
        class GoodSocket:
            def __init__(self):
                self.buffer = []

            def sendall(self, data):
                self.buffer.append(data)

            def close(self):
                return None

        class FailingSocket:
            def sendall(self, data):
                raise OSError("target dropped")

            def close(self):
                return None

        server = PythonArenaServer("127.0.0.1", 5050, db_path=":memory:")
        challenger = UserSession(address=("127.0.0.1", 1), socket=GoodSocket(), username="Teo")
        target = UserSession(address=("127.0.0.1", 2), socket=FailingSocket(), username="Joey")
        server.user_registry.register("Teo", challenger)
        server.user_registry.register("Joey", target)
        server.lobby_manager.set_waiting("Teo")
        server.handle_challenge_player(challenger, {"target_username": "Joey"})
        self.assertIsNotNone(server.user_registry.get_session("Teo"))
        self.assertIsNone(server.lobby_manager.pending_challenger_for("Joey"))
        self.assertTrue(server.lobby_manager.is_waiting("Teo"))
        self.assertGreaterEqual(len(challenger.socket.buffer), 2)
        outbound_messages = [decode_message(item[4:]) for item in challenger.socket.buffer]
        self.assertNotIn(message_types.CHALLENGE_PLAYER, [message["type"] for message in outbound_messages])
        self.assertIn(message_types.ERROR, [message["type"] for message in outbound_messages])

    def test_watch_match_subscribes_spectator_and_sends_state(self):
        class FakeSocket:
            def __init__(self):
                self.buffer = []

            def sendall(self, data):
                self.buffer.append(data)

            def close(self):
                return None

        server = PythonArenaServer("127.0.0.1", 5050, db_path=":memory:")
        spectator = UserSession(address=("127.0.0.1", 3), socket=FakeSocket(), username="Sara")
        server.user_registry.register("Sara", spectator)
        server.active_match = Match(players=["Teo", "Joey"])
        server.handle_watch_match(spectator)
        self.assertIn("Sara", server.get_spectators())
        self.assertEqual(len(spectator.socket.buffer), 2)
        first = decode_message(spectator.socket.buffer[0][4:])
        second = decode_message(spectator.socket.buffer[1][4:])
        self.assertEqual(first["type"], message_types.WATCH_MATCH)
        self.assertEqual(second["type"], message_types.MATCH_START)
        self.assertTrue(second["payload"]["spectator"])

    def test_handle_cheer_updates_match_state(self):
        class FakeSocket:
            def __init__(self):
                self.buffer = []

            def sendall(self, data):
                self.buffer.append(data)

            def close(self):
                return None

        server = PythonArenaServer("127.0.0.1", 5050, db_path=":memory:")
        teo = UserSession(address=("127.0.0.1", 1), socket=FakeSocket(), username="Teo")
        joey = UserSession(address=("127.0.0.1", 2), socket=FakeSocket(), username="Joey")
        server.user_registry.register("Teo", teo)
        server.user_registry.register("Joey", joey)
        server.active_match = Match(players=["Teo", "Joey"])
        server.handle_cheer(teo, {"text": "Nice move"})
        self.assertEqual(server.active_match.cheers[-1]["text"], "Nice move")

    def test_start_match_includes_countdown_seconds(self):
        class FakeSocket:
            def __init__(self):
                self.buffer = []

            def sendall(self, data):
                self.buffer.append(data)

            def close(self):
                return None

        server = PythonArenaServer("127.0.0.1", 5050, db_path=":memory:")
        teo = UserSession(address=("10.0.0.1", 1111), socket=FakeSocket(), username="Teo")
        joey = UserSession(address=("10.0.0.2", 2222), socket=FakeSocket(), username="Joey")
        server.user_registry.register("Teo", teo)
        server.user_registry.register("Joey", joey)
        with mock.patch("server.server.threading.Thread") as patched_thread:
            patched_thread.return_value.start = lambda: None
            started, reason = server.start_match(["Teo", "Joey"])
        self.assertTrue(started)
        self.assertIsNone(reason)
        first_match_start = decode_message(teo.socket.buffer[0][4:])
        self.assertEqual(first_match_start["type"], message_types.MATCH_START)
        self.assertEqual(first_match_start["payload"]["countdown_seconds"], 3)

    def test_start_match_sends_chat_peer_info(self):
        class FakeSocket:
            def __init__(self):
                self.buffer = []

            def sendall(self, data):
                self.buffer.append(data)

            def close(self):
                return None

        server = PythonArenaServer("127.0.0.1", 5050, db_path=":memory:")
        teo = UserSession(address=("10.0.0.1", 1111), socket=FakeSocket(), username="Teo", chat_port=7001)
        joey = UserSession(address=("10.0.0.2", 2222), socket=FakeSocket(), username="Joey", chat_port=7002)
        server.user_registry.register("Teo", teo)
        server.user_registry.register("Joey", joey)
        with mock.patch("server.server.threading.Thread") as patched_thread:
            patched_thread.return_value.start = lambda: None
            started, reason = server.start_match(["Teo", "Joey"])
        self.assertTrue(started)
        self.assertIsNone(reason)
        decoded_messages = [decode_message(item[4:]) for item in teo.socket.buffer]
        chat_messages = [message for message in decoded_messages if message["type"] == message_types.CHAT_PEER_INFO]
        self.assertEqual(len(chat_messages), 1)
        self.assertEqual(chat_messages[0]["payload"]["peer_port"], 7002)

In [ ]:
# Future feature tests should be added as new unittest.TestCase classes above this cell.
suite = unittest.TestSuite()
for obj in list(globals().values()):
    if isinstance(obj, type) and issubclass(obj, unittest.TestCase):
        suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(obj))
result = unittest.TextTestRunner(verbosity=2).run(suite)
if not result.wasSuccessful():
    raise SystemExit(1)
print("All notebook tests passed.")